In [1]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "data"
qa_dataset_path = DATASET_DIR / "raw" / "nitibench" / "data" / "nitibench-ccl.parquet"
qa_dataset = pd.read_parquet(qa_dataset_path)

In [8]:
qa_dataset.head(2)

,question,answer,relevant_laws,reference_answer,reference_laws
0,ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญ...,ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินส...,[{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน...,ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินส...,[{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน...
1,ถ้าผู้ิยู่ในปกครองได้ยินยอมในการกระทำของผู้ปกค...,การที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นไม่ได้คุ...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",การที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นไม่ได้คุ...,[]


In [9]:
# Create Law Name col extract from relevant laws col
# Value Count in Law col each Law Name use X % of value count
# Extract Row each Law Name X % from value count
# Second same proportion but qa will imbanlance + Min Cap
# $$Count_{law} = \max(MinCap, \min(MaxCap, Total \times Proportion))$$

qa_dataset["law_name"] = qa_dataset["relevant_laws"].apply(lambda row: row[0].get("law_name"))
proportion_qa_dict = qa_dataset["law_name"].value_counts().to_dict()
print("Total QA:", sum(proportion_qa_dict.values()))


new_proportion_qa_dict = {}
proportion_per = 0.4

def calculate_qa_count(total_law, proportion):
    #qa total in final
    MinCap = min(20, total_law)
    MaxCap = 0.15 * (total_law * proportion)
    return int(max(MinCap, min(MaxCap, total_law * proportion)))

# # Count = max ( MinCap, min(MaxCap, Total * Proportion))
for key, value in proportion_qa_dict.items():
    new_proportion_qa_dict[key] = calculate_qa_count(total_law=value, proportion=proportion_per)

print(new_proportion_qa_dict)
print("New Total QA:", sum(new_proportion_qa_dict.values()))

Total QA: 3729
{'ประมวลกฎหมายแพ่งและพาณิชย์': 97, 'ประมวลรัษฎากร': 29, 'พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535': 20, 'พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535': 20, 'พระราชบัญญัติธุรกิจสถาบันการเงิน พ.ศ. 2551': 20, 'พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550': 20, 'พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514': 20, 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546': 20, 'พระราชกำหนดการประกอบธุรกิจสินทรัพย์ดิจิทัล พ.ศ. 2561': 20, 'พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558': 20, 'พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ.ศ. 2550': 20, 'พระราชบัญญัติวิชาชีพบัญชี พ.ศ. 2547': 20, 'พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535': 20, 'พระราชบัญญัติกำหนดความผิดเกี่ยวกับห้างหุ้นส่วนจดทะเบียน ห้างหุ้นส่วนจำกัด บริษัทจำกัด สมาคม และมูลนิธิ พ.ศ. 2499': 20, 'พระราชบัญญัติสมาคมการค้า พ.ศ. 2509': 20, 'พระราชบัญญัติการบัญชี พ.ศ. 2543': 20, 'พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542': 20, 'พระราชบัญญัติกองทุนสำรองเลี้ยงชีพ พ.ศ. 2530': 20, 'พระราชกำหนดนิติบุคคลเฉพาะกิจเพื่อกา

In [ ]:
# To create a copy of a pandas DataFrame while removing all records
new_test_df = qa_dataset[:0].drop(columns=["law_name"])

for law_name in new_proportion_qa_dict.keys():
    count = new_proportion_qa_dict[law_name]

    # 1. แยกกลุ่มที่มี Ref Law (ไม่ว่าง) และ ไม่มี Ref Law (ว่าง) ออกจากกัน
    full_law_df = qa_dataset[qa_dataset["law_name"] == law_name]
    
    has_ref_df = full_law_df[full_law_df["reference_laws"].apply(len) > 0]
    no_ref_df = full_law_df[full_law_df["reference_laws"].apply(len) == 0]

    # 2. เลือกจากกลุ่มที่มี Ref ก่อน
    if len(has_ref_df) >= count:
        # ถ้ามีกลุ่มมี Ref มากพอ ให้สุ่มเลือกมาตามจำนวน count (ใช้ sample เพื่อความหลากหลายแทน head)
        filter_by_law_df = has_ref_df.sample(n=count, random_state=42)
    else:
        # ถ้ากลุ่มมี Ref ไม่พอ ให้เอามาทั้งหมด แล้วไปดึงจากกลุ่มที่ไม่มี Ref มาเพิ่มให้ครบ
        needed = count - len(has_ref_df)
        # ดึงจากกลุ่มไม่มี Ref เท่าที่ต้องการ (หรือเท่าที่มีอยู่จริง)
        extra_df = no_ref_df.sample(n=min(needed, len(no_ref_df)), random_state=42)
        filter_by_law_df = pd.concat([has_ref_df, extra_df])
    
    # concat เข้า new_test_df 
    new_test_df = pd.concat([new_test_df, filter_by_law_df], ignore_index=True)

In [2]:
test_qa_dataset_path = DATASET_DIR / "tests" /  "test_dataset_2026-04-01.parquet"
new_test_df = pd.read_parquet(test_qa_dataset_path)

In [3]:
exclude_indices = [99, 115, 119, 136, 180, 181, 208, 218, 403]
new_test_df_delete_noise = new_test_df.drop(exclude_indices)
new_test_df_delete_noise.head(2)

,question,answer,relevant_laws,reference_answer,reference_laws,law_name
0,การเเสดงเจตนาตัดมิให้รับมรดกจะถอนได้หรือไม่,ได้ ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1609 กา...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ได้,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
1,การแปรสภาพห้างหุ้นส่วนจำกัดเป็นบริษัทจำกัด ต้อ...,หุ้นส่วนผู้จัดการต้องส่งมอบกิจการ ทรัพย์สิน บั...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ภายใน 14 วันนับแต่วันที่ผู้เป็นหุ้นส่วนได้ให้ค...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์


In [4]:
total_new_qa = int(new_test_df_delete_noise["law_name"].value_counts().sum())
total_new_qa

# ดู QA ที่มีการอ้างอิง Ref
empty_ref_count = int((new_test_df_delete_noise["reference_laws"].apply(len) == 0).sum())
not_empty_ref_count = total_new_qa - empty_ref_count
print("empty_ref_count:", empty_ref_count)
print("not_empty_ref_count:", not_empty_ref_count)

empty_ref_count: 44
not_empty_ref_count: 444


In [ ]:
import pandas as pd
import numpy as np

# 1. สร้าง Column โดยระบุว่าต้องมี 2 กลุ่มนี้เสมอ (ป้องกัน Error ถ้าข้อมูลมาไม่ครบ)
categories = ['Reference QA', 'Simple QA']
law_type = np.where(new_test_df_delete_noise['reference_laws'].apply(len) == 0, 'Simple QA', 'Reference QA')
new_test_df_delete_noise['Law_Type'] = pd.Categorical(law_type, categories=categories)

# 2. ใช้ crosstab สร้างตารางพร้อมผลรวม (สั้นมาก!)
pivot_result = pd.crosstab(
    new_test_df_delete_noise['law_name'], 
    new_test_df_delete_noise['Law_Type'], 
    margins=True, 
    margins_name='Total'
).reset_index()

# 3. ล้างหัวตารางที่เกินมา
pivot_result.columns.name = None

# ตั้งค่าให้แสดงข้อความในคอลัมน์แบบไม่ตัด (None คือไม่จำกัดความยาว)
pd.set_option('display.max_colwidth', None)

pivot_result

,law_name,Reference QA,Simple QA,Total
0,ประมวลกฎหมายแพ่งและพาณิชย์,97,0,97
1,ประมวลรัษฎากร,26,0,26
2,พระราชกำหนดการประกอบธุรกิจสินทรัพย์ดิจิทัล พ.ศ. 2561,20,0,20
3,พระราชกำหนดนิติบุคคลเฉพาะกิจเพื่อการแปลงสินทรัพย์เป็นหลักทรัพย์ พ.ศ. 2540,13,7,20
4,พระราชบัญญัติกองทุนสำรองเลี้ยงชีพ พ.ศ. 2530,13,7,20
5,พระราชบัญญัติการบัญชี พ.ศ. 2543,18,1,19
6,พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550,20,0,20
7,พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542,15,5,20
8,พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535,20,0,20
9,พระราชบัญญัติกำหนดความผิดเกี่ยวกับห้างหุ้นส่วนจดทะเบียน ห้างหุ้นส่วนจำกัด บริษัทจำกัด สมาคม และมูลนิธิ พ.ศ. 2499,20,0,20


In [ ]:
pd.reset_option('all')

In [11]:
import pandas as pd

# 1. โหลดข้อมูล (สมมติว่าไฟล์อยู่ในเครื่อง)
df = new_test_df_delete_noise

# 2. คำนวณจำนวน Reference Laws ในแต่ละแถว
# ใช้ .apply(len) เพื่อดูจำนวนสมาชิกใน list
df['ref_count'] = df['reference_laws'].apply(lambda x: len(x))

# 3. สร้างตารางสรุป
summary_df = df['ref_count'].value_counts().sort_index().reset_index()
summary_df.columns = ['จำนวน Reference Laws (ชุด)', 'จำนวนข้อ (QA)']

# 4. คำนวณร้อยละ (%)
total_qa = len(df)
summary_df['คิดเป็นร้อยละ (%)'] = (summary_df['จำนวนข้อ (QA)'] / total_qa * 100).map('{:.2f}%'.format)

# 5. เพิ่มแถวรวม (Optional)
total_row = pd.DataFrame({
    'จำนวน Reference Laws (ชุด)': ['รวมทั้งหมด'],
    'จำนวนข้อ (QA)': [total_qa],
    'คิดเป็นร้อยละ (%)': ['100.00%']
})

final_table = pd.concat([summary_df, total_row], ignore_index=True)

final_table

,จำนวน Reference Laws (ชุด),จำนวนข้อ (QA),คิดเป็นร้อยละ (%)
0,0,44,9.02%
1,1,262,53.69%
2,2,95,19.47%
3,3,34,6.97%
4,4,20,4.10%
5,5,11,2.25%
6,6,5,1.02%
7,7,5,1.02%
8,8,4,0.82%
9,9,2,0.41%


In [5]:
new_test_df.head(2)

,question,answer,relevant_laws,reference_answer,reference_laws,law_name
0,การเเสดงเจตนาตัดมิให้รับมรดกจะถอนได้หรือไม่,ได้ ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1609 กา...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ได้,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
1,การแปรสภาพห้างหุ้นส่วนจำกัดเป็นบริษัทจำกัด ต้อ...,หุ้นส่วนผู้จัดการต้องส่งมอบกิจการ ทรัพย์สิน บั...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ภายใน 14 วันนับแต่วันที่ผู้เป็นหุ้นส่วนได้ให้ค...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์


In [ ]:
# df.to_parquet(DATASET_PATH, engine='pyarrow', index=False)
test_qa_dataset_path = DATASET_DIR / "tests" /  "test_dataset_2026-04-01.parquet"
new_test_df.to_parquet(test_qa_dataset_path, engine='pyarrow', index=False)

In [2]:
test_qa_dataset_path = DATASET_DIR / "tests" /  "test_dataset_2026-04-01.parquet"
test_qa_dataset = pd.read_parquet(test_qa_dataset_path)
test_qa_dataset_filter =  test_qa_dataset.iloc[:5]

In [3]:
test_qa_dataset_filter

,question,answer,relevant_laws,reference_answer,reference_laws,law_name
0,การเเสดงเจตนาตัดมิให้รับมรดกจะถอนได้หรือไม่,ได้ ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1609 กา...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ได้,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
1,การแปรสภาพห้างหุ้นส่วนจำกัดเป็นบริษัทจำกัด ต้อ...,หุ้นส่วนผู้จัดการต้องส่งมอบกิจการ ทรัพย์สิน บั...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ภายใน 14 วันนับแต่วันที่ผู้เป็นหุ้นส่วนได้ให้ค...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
2,ซื้อหวยใต้ดิน แล้วเขาไม่จ่าย ฟ้องได้ไหม,ไม่ได้ เพราะมีแค่เพียง สลากกินแบ่งของรัฐบาลเท่...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ไม่ได้ เพราะมีแค่เพียง สลากกินแบ่งของรัฐบาลเท่...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
3,พ่อแม่โดยกำเนิดกลับมามีอำนาจปกครองเมื่อไร,นับแต่เวลาที่ผู้รับบุตรบุญธรรมตาย หรือนับแต่เว...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",นับแต่เวลาที่ผู้รับบุตรบุญธรรมตาย หรือนับแต่เว...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์
4,ระยะเวลาในการฟ้องคดีมรดกมีหลักเกณฑ์อย่างไร,ห้ามมิให้ฟ้องคดีมรดกเมื่อพ้นกำหนดหนึ่งปี นับแต...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ห้ามมิให้ฟ้องคดีมรดกเมื่อพ้นกำหนดหนึ่งปี นับแต...,"[{'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 's...",ประมวลกฎหมายแพ่งและพาณิชย์


In [15]:
test_qa_dataset_filter_path = DATASET_DIR / "tests" /  "test_dataset_2026-04-01_filter.parquet"
test_qa_dataset_filter.to_parquet(test_qa_dataset_filter_path, engine='pyarrow', index=False)